In [ ]:
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional
# --- 用路径提 seed（支持 seed2 / seed-2 / seed_2 / SEED2 等）---
def extract_seed_from_path(path: str) -> str:
    m = re.search(r"(?:^|[\\/])seed[_-]?(\d+)(?:[\\/]|$)", path, flags=re.IGNORECASE)
    return m.group(1) if m else "unknown"

# --- 用路径兜底提取 model（匹配 .../<model>/seedX/... 的 <model>）---
def extract_model_from_path(path: str) -> Optional[str]:
    parts = Path(path).parts
    # 找到 'seedX' 这个组件在路径中的位置，取它的上一个作为 model
    for i, comp in enumerate(parts):
        if re.fullmatch(r"seed[_-]?\d+", comp, flags=re.IGNORECASE):
            if i - 1 >= 0:
                return parts[i - 1]
    return None

In [ ]:
# --- Creativity @ answer-level: subtask × model panels, colored by generation, markers by seed ---
import os, re, math
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.collections import LineCollection

# ======= 配置：两张 index CSV（里面有 csv_file_path 列）======
INDEX_FILES = {
    "Summarization": "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sum_raw.csv",
    "Simplification": "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sim_raw.csv",
}

# 只画 child
CHILD_ONLY = False

# 列名候选
PATH_COL_CANDS  = ["csv_file_path", "csv_path", "path"]
GEN_COL_CANDS   = ["generation", "gen", "g"]
TYPE_COL_CANDS  = ["type", "sample_type", "role"]
FITN_COL_CANDS  = ["fitness_normed"]

# ✅ 改：优先找“原始多样性/距离”列，其次才用已经归一化的列兜底
DIVERSITY_RAW_CANDS = [
    "total_distance", "novelty", "near_distance", "diversity", "distance"
]
DIVERSITY_NORMED_FALLBACK = [
    "total_distance_normed", "novelty_normed", "near_distance_normed"
]

MODEL_COL_CANDS = ["model", "alias", "model_alias", "price_key"]
SEED_COL_CANDS  = ["seed", "seed_id", "run"]

# 视觉：白底、紧凑
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use("default")
mpl.rcParams["figure.facecolor"]  = "white"
mpl.rcParams["axes.facecolor"]    = "white"
mpl.rcParams["savefig.facecolor"] = "white"

# seed → marker
SEED_MARKERS = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', '*', 'h', 'H', 'p']
def seed_to_marker(seed_value) -> str:
    try:
        i = int(seed_value)
    except Exception:
        return 'o'
    return SEED_MARKERS[i % len(SEED_MARKERS)]

def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def find_first_column(df: pd.DataFrame, cands: List[str]) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    return None

def try_get_seed_series(df: pd.DataFrame, path: str) -> pd.Series:
    for c in SEED_COL_CANDS:
        if c in df.columns:
            return df[c]
    name = Path(path).name
    m = re.search(r"seed[_\-]?(\d+)", name, flags=re.IGNORECASE)
    if m:
        v = m.group(1)
        return pd.Series([int(v)]*len(df), index=df.index)
    return pd.Series(["unknown"]*len(df), index=df.index)

def load_index(index_path: str) -> pd.DataFrame:
    p = Path(index_path)
    idx = pd.read_csv(p)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    for c in MODEL_COL_CANDS + SEED_COL_CANDS:
        if c in idx.columns:
            keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_process_csv(process_csv_path: str) -> Optional[pd.DataFrame]:
    p = Path(process_csv_path)
    if not p.exists():
        print(f"[warn] missing process CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def pareto_front_xy(x: np.ndarray, y: np.ndarray, direction: str = "ltr") -> np.ndarray:
    ok = np.isfinite(x) & np.isfinite(y)
    if not ok.any():
        return np.empty((0, 2))
    xv = x[ok]; yv = y[ok]
    if direction == "rtl":
        order = np.argsort(xv)[::-1]          # x 降序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)
    else:
        order = np.argsort(xv)                 # x 升序
        xv = xv[order]; yv = yv[order]
        best = -np.inf
        pts = []
        for xi, yi in zip(xv, yv):
            if yi > best:
                pts.append((xi, yi)); best = yi
        return np.array(pts)

# ======= 第一步：读取并收集“原始多样性值” =======
panels: Dict[Tuple[str,str], pd.DataFrame] = {}
all_div_raw = []      # 收集全局的 raw diversity 值
used_fallback = False # 是否使用了已归一化列作为原始兜底

for subtask, index_csv in INDEX_FILES.items():
    idx = load_index(index_csv)
    for _, row in idx.iterrows():
        path = str(row["csv_file_path"]).strip()
        df = read_process_csv(path)
        if df is None or df.empty:
            continue

        gen_col  = find_col(df, GEN_COL_CANDS, required=True)
        fit_col  = find_col(df, FITN_COL_CANDS, required=True)

        # 优先找原始列；找不到就用已归一化列兜底
        div_raw_col = find_first_column(df, DIVERSITY_RAW_CANDS)
        if div_raw_col is None:
            div_raw_col = find_first_column(df, DIVERSITY_NORMED_FALLBACK)
            if div_raw_col is None:
                print(f"[warn] {path} has no diversity/novelty column (raw or normed).")
                continue
            used_fallback = True

        type_col = find_col(df, TYPE_COL_CANDS, required=False)
        model_col= find_col(df, MODEL_COL_CANDS, required=False)

        # 仅 child
        if CHILD_ONLY and type_col is not None:
            df = df[df[type_col].astype(str).str.lower().eq("child")].copy()
        if df.empty:
            continue

        seed_series = try_get_seed_series(df, path)

        use_cols = [gen_col, fit_col, div_raw_col]
        if model_col: use_cols.append(model_col)
        sub = df[use_cols].copy()
        sub.rename(columns={
            gen_col      : "generation",
            fit_col      : "fitness_normed",
            div_raw_col  : "diversity_raw",   # 👈 统一命名为“原始多样性值”
        }, inplace=True)
        sub["seed"] = seed_series.values
        sub["subtask"] = subtask

        # 模型名（尽力拿）
        if model_col and model_col in df.columns and df[model_col].notna().any():
            model_name = str(df[model_col].dropna().iloc[0])
        else:
            model_name = None
            for c in MODEL_COL_CANDS:
                if c in row and isinstance(row[c], str) and row[c].strip():
                    model_name = row[c]; break
            if not model_name:
                model_name = Path(path).stem

        key = (subtask, model_name)
        panels.setdefault(key, pd.DataFrame())
        panels[key] = pd.concat([panels[key], sub], ignore_index=True)

        # 收集全局 raw
        all_div_raw.append(pd.to_numeric(sub["diversity_raw"], errors="coerce"))

# ======= 第二步：按 subtask 归一（也保留全局可选） =======
NORM_SCOPE = "by_subtask"   # 可选: "global" / "by_subtask" / "by_panel"
ROBUST_Q = (0, 99)            # 例如 (1, 99) 走分位数鲁棒归一；None 则用 min–max

# 先做清洗（与原来一致），并把 panels 里的 df 统一成数值
cleaned_panels = {}
for k, df in list(panels.items()):
    if df.empty:
        continue
    df["generation"]      = pd.to_numeric(df["generation"], errors="coerce")
    df["fitness_normed"]  = pd.to_numeric(df["fitness_normed"], errors="coerce")
    df["diversity_raw"]   = pd.to_numeric(df["diversity_raw"],  errors="coerce")
    df = df.dropna(subset=["generation", "fitness_normed", "diversity_raw"])
    df = df.sort_values("generation").reset_index(drop=True)
    cleaned_panels[k] = df
panels = cleaned_panels

if not panels:
    print("[warn] no diversity values found; abort plotting.")
else:
    # 根据归一范围，收集 raw 值并计算 min/max
    if NORM_SCOPE == "global":
        vec = pd.concat([df["diversity_raw"] for df in panels.values()], ignore_index=True).astype(float)
        if ROBUST_Q:
            lo, hi = np.nanpercentile(vec, ROBUST_Q)
        else:
            lo, hi = np.nanmin(vec), np.nanmax(vec)
        den = (hi - lo) if np.isfinite(hi) and np.isfinite(lo) and hi > lo else 1.0

        for k, df in panels.items():
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    elif NORM_SCOPE == "by_subtask":
        # k = (subtask, model_name)
        per_subtask_vals = defaultdict(list)
        for (subtask, _m), df in panels.items():
            per_subtask_vals[subtask].append(df["diversity_raw"])

        # 计算各 subtask 的 [lo, hi]
        subtask_minmax = {}
        for subtask, series_list in per_subtask_vals.items():
            vec = pd.concat(series_list, ignore_index=True).astype(float)
            if ROBUST_Q:
                lo, hi = np.nanpercentile(vec, ROBUST_Q)
            else:
                lo, hi = np.nanmin(vec), np.nanmax(vec)
            subtask_minmax[subtask] = (lo, hi)

        # 写回归一后的列
        for (subtask, _m), df in panels.items():
            lo, hi = subtask_minmax[subtask]
            den = (hi - lo) if np.isfinite(hi) and np.isfinite(lo) and hi > lo else 1.0
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    elif NORM_SCOPE == "by_panel":
        # 每个 (subtask, model) 面板单独归一
        for k, df in panels.items():
            vec = df["diversity_raw"].astype(float)
            if ROBUST_Q:
                lo, hi = np.nanpercentile(vec, ROBUST_Q)
            else:
                lo, hi = np.nanmin(vec), np.nanmax(vec)
            den = (hi - lo) if np.isfinite(hi) and np.isfinite(lo) and hi > lo else 1.0
            df["diversity_gnormed"] = ((df["diversity_raw"] - lo) / den).clip(0.0, 1.0)

    else:
        raise ValueError(f"Unknown NORM_SCOPE: {NORM_SCOPE}")

# ======= 第三步：画所有 (subtask, model) 子图 =======
keys = sorted(panels.keys(), key=lambda x: (x[0], str(x[1]).lower()))
n = len(keys)
if n == 0:
    print("[warn] no data to plot.")
else:
    ncols = min(5, n)
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0*ncols, 2.6*nrows), squeeze=False)
    fig.suptitle("Prompt Optimization(Simplification/Summarization) - Novelty at the Answer Level",
                 fontsize=16, y=0.995)

    # generation 全局范围用于 colorbar
    g_all = pd.concat([panels[k]["generation"] for k in keys], ignore_index=True)
    gmin_all, gmax_all = g_all.min(), g_all.max()
    cmap = plt.get_cmap("viridis")
    gnorm_global = Normalize(vmin=gmin_all, vmax=gmax_all)

    for i, key in enumerate(keys):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        df = panels[key]
        subtask, model = key

        # 颜色：按全局 generation 范围映射
        colors = cmap(gnorm_global(df["generation"].to_numpy()))

        # seed 分 marker
        seeds = df["seed"].astype(str).fillna("unknown")
        uniq_seeds = seeds.unique().tolist()

        for s in uniq_seeds:
            mask = seeds.eq(s).to_numpy()
            mk = seed_to_marker(s)
            ax.scatter(df.loc[mask, "fitness_normed"], df.loc[mask, "diversity_gnormed"],
                       s=18, marker=mk, c=colors[mask], edgecolor="white", linewidth=0.35, alpha=0.95, zorder=2)

        # # —— 中位数进化轨迹（跨 seed 聚合；使用全局归一后的 y）——
        # g_med = (df.groupby('generation', as_index=False)
        #            .agg(fit_med=('fitness_normed', 'median'),
        #                 nov_med=('diversity_gnormed', 'median'))
        #            .sort_values('generation'))
        # if len(g_med) >= 2:
        #     ax.plot(g_med['fit_med'], g_med['nov_med'],
        #             color='lightsalmon', lw=1.6, alpha=0.85, zorder=3, label='_nolegend_')
        #     ax.scatter(g_med.iloc[[0]]['fit_med'],  g_med.iloc[[0]]['nov_med'],
        #                s=24, marker='o', facecolor='none', edgecolor='black', lw=0.7, zorder=3)
        #     ax.scatter(g_med.iloc[[-1]]['fit_med'], g_med.iloc[[-1]]['nov_med'],
        #                s=28, marker='*', facecolor='black', edgecolor='white', lw=0.5, zorder=3)

        # —— Pareto frontier（x=fitness_normed, y=diversity_gnormed；从右往左）——
        pts = pareto_front_xy(
            df["fitness_normed"].to_numpy(float),
            df["diversity_gnormed"].to_numpy(float),
            direction="rtl"
        )
        if pts.size:
            ax.plot(pts[:,0], pts[:,1], '-', color="steelblue", lw=1.5, alpha=0.85, zorder=1.5)

        # 轴：固定 0~1，白底，淡网格
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.grid(True, color="#eee", linewidth=0.8, zorder=0)

        # 只在最左列/最下行显示坐标轴标签
        if c == 0:
            ax.set_ylabel("Normalized diversity")
        else:
            ax.set_yticklabels([])
        if r == nrows - 1:
            ax.set_xlabel("Normalized fitness")
        else:
            ax.set_xticklabels([])

        # 标题
        ax.set_title(f"{subtask} | {model}", fontsize=9.5, pad=3)

    # 关闭多余子图
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r][c].set_axis_off()

    # 全局 generation colorbar（右侧）
    cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gnorm_global)
    cb.set_label("Generation", rotation=90)

    fig.subplots_adjust(left=0.06, right=0.90, top=0.94, bottom=0.08, wspace=0.16, hspace=0.20)
    plt.savefig("/Users/lievretre/Desktop/evo_eval_plots_10_28/novelty_trajectory_plots/novelty_scatter_promptopt.png",dpi=300)
    plt.show()

In [ ]:
 === TSP — Task-level MDS (distance= 1 - |E∩|/n)  ===
# 输入：多个 subtask 的 index.csv（每个 index 里列出要遍历的 CSV）
# 输出：每个 (subtask, model) 一个面板，颜色=generation，大小=fitness

import os, re, math, ast
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from sklearn.manifold import MDS

# -------------------------
# 你的 index 文件字典
# -------------------------
INDEX_FILES = {
    "Summarization": "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sum_raw.csv",
    "Simplification": "/Users/lievretre/Desktop/evo_eval_sheets/promptopt_sim_raw.csv",
}

# 只画 child（TSP 场景一般 parent/child 都有，先给开关）
CHILD_ONLY = False

# 列名候选（自动探测）
PATH_COL_CANDS  = ["csv_file_path", "csv_path", "path"]
GEN_COL_CANDS   = ["generation", "gen", "g"]
TYPE_COL_CANDS  = ["type", "sample_type", "role"]
FITN_COL_CANDS  = ["fitness_normed", "fitness", "score"]
MODEL_COL_CANDS = ["model", "alias", "model_alias", "price_key", "engine"]
SEED_COL_CANDS  = ["seed", "seed_id", "run"]

# genome 列名候选
TOUR_COL_CANDS  = ["genome", "path", "perm", "route", "cycle"]

# MDS 超参
MDS_MAX_POINTS   = 4000      # 超过则抽样学平面 + OOS 插值
PER_BUCKET       = 60        # (model, generation) 每桶上限（用于抽样）
MDS_KW = dict(
    n_components=2,
    dissimilarity="precomputed",
    n_init=1,
    max_iter=300,
    eps=1e-3,
    random_state=42,
    verbose=1,
)

# OOS 插值：KNN & 权重
OOS_K = 8
OOS_P = 2.0     # 权重=1/(d+1e-8)^P
RNG = np.random.default_rng(42)

# 视觉
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use("default")
mpl.rcParams["figure.facecolor"]  = "white"
mpl.rcParams["axes.facecolor"]    = "white"
mpl.rcParams["savefig.facecolor"] = "white"

# ========== 通用工具 ==========
def find_col(df: pd.DataFrame, cands: List[str], required: bool=True) -> Optional[str]:
    for c in cands:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Required column not found. Tried: {cands}")
    return None

def load_index(index_path: str) -> pd.DataFrame:
    idx = pd.read_csv(index_path)
    path_col = find_col(idx, PATH_COL_CANDS, required=True)
    keep = [path_col]
    for c in MODEL_COL_CANDS + SEED_COL_CANDS:
        if c in idx.columns: keep.append(c)
    out = idx[keep].copy()
    out.rename(columns={path_col: "csv_file_path"}, inplace=True)
    return out

def read_csv_maybe(path: str) -> Optional[pd.DataFrame]:
    p = Path(path)
    if not p.exists():
        print(f"[warn] missing CSV: {p}")
        return None
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[warn] read failed: {p} -> {e}")
        return None

def coerce_tour(x):
    """把 tour 解析成 list[int]"""
    if isinstance(x, list): return [int(v) for v in x]
    if isinstance(x, np.ndarray): return [int(v) for v in x.tolist()]
    if isinstance(x, str):
        try:
            v = ast.literal_eval(x)
            return [int(t) for t in (v.tolist() if isinstance(v, np.ndarray) else list(v))]
        except Exception:
            # 兜底：逗号/空格切
            toks = re.findall(r"\d+", x)
            return [int(t) for t in toks]
    raise ValueError(f"Unsupported tour type: {type(x)}")

def edge_indexer(n_cities: int):
    """
    返回：map[(i,j)] -> col_id  (i<j)
    用于把无向边映射到 0..E-1 的列索引
    """
    idx = {}
    col = 0
    for i in range(n_cities):
        for j in range(i+1, n_cities):
            idx[(i,j)] = col
            col += 1
    return idx, col  # col == E = n*(n-1)/2

def tours_to_incidence(tours, n_cities: int, edge_map):
    """
    tours: List[List[int]]
    返回 A: uint8 矩阵，形状 (S, E)，每行有 n_cities 个 1（闭环）
    """
    S = len(tours)
    E = len(edge_map)
    A = np.zeros((S, E), dtype=np.uint8)
    for r, t in enumerate(tours):
        m = len(t)
        # 闭环
        for k in range(m):
            a = int(t[k]); b = int(t[(k+1) % m])
            if a == b: 
                continue
            if a > b: 
                a, b = b, a
            A[r, edge_map[(a,b)]] = 1
    return A

def distances_on_fit(A_fit: np.ndarray, n_edges: int) -> np.ndarray:
    """
    A_fit: (m, E) uint8
    返回 D_fit: (m, m) float32，D = 1 - (|E∩| / n_edges)
    """
    # 交集大小 = 点积；用 int32/float32 计算
    # 注意：A_fit 是 0/1，A_fit @ A_fit.T 非常快（BLAS）
    inter = (A_fit @ A_fit.T).astype(np.float32)
    D = 1.0 - inter / float(n_edges)
    # 数值安全
    np.fill_diagonal(D, 0.0)
    return D
    
def distance_one_to_many_edges(Ei: frozenset, E_list: List[frozenset], n_edges: int) -> np.ndarray:
    """给一个边集，计算到一批边集的边差距离"""
    out = np.empty(len(E_list), dtype=np.float32)
    for k, Ej in enumerate(E_list):
        inter = len(Ei.intersection(Ej))
        out[k] = 1.0 - (inter / max(1, n_edges))
    return out

def robust_minmax(x: np.ndarray, q=(1,99)):
    lo, hi = np.nanpercentile(x, q)
    den = (hi-lo) if hi>lo else 1.0
    return np.clip((x-lo)/den, 0, 1)

def stratified_sample_idx(df: pd.DataFrame, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS) -> np.ndarray:
    parts = []
    for (m, g), sub in df.groupby(["model","generation"], sort=False):
        if len(sub) > per_bucket:
            parts.append(sub.sample(per_bucket, random_state=int(RNG.integers(1<<31))))
        else:
            parts.append(sub)
    out = pd.concat(parts, ignore_index=False)
    if len(out) > total_cap:
        out = out.sample(total_cap, random_state=42)
    return out.index.to_numpy()

def oos_place_by_blocks(A_all, fit_idx, Y_fit, n_edges, k=8, p=2.0, block=4000):
    """
    A_all: (N, E)  全体 0/1 发生矩阵
    fit_idx: (m,)  抽样索引
    Y_fit: (m,2)   MDS 基点坐标
    返回：Y: (N,2)
    """
    N = A_all.shape[0]
    m = len(fit_idx)
    Y = np.zeros((N, 2), dtype=np.float32)
    Y[fit_idx] = Y_fit

    not_mask = np.ones(N, dtype=bool)
    not_mask[fit_idx] = False
    q_idx = np.where(not_mask)[0]
    if len(q_idx) == 0:
        return Y

    A_fit = A_all[fit_idx]                   # (m, E)
    A_fitT = A_fit.T.astype(np.uint8, copy=False)

    for s in range(0, len(q_idx), block):
        sl = q_idx[s: s+block]               # 这一块查询
        Aq  = A_all[sl]                      # (b, E)
        # 交集数 = Aq @ A_fit^T
        inter = (Aq @ A_fitT).astype(np.float32)   # (b, m)
        Dq = 1.0 - inter / float(n_edges)         # (b, m)

        # 取每行的 k 个最近邻（最小距离）
        if k < m:
            # argpartition 更快
            nn = np.argpartition(Dq, kth=k, axis=1)[:, :k]   # (b, k)
            rows = np.arange(len(sl))[:, None]
            dnn = Dq[rows, nn]                               # (b, k)
        else:
            nn = np.tile(np.arange(m), (len(sl),1))
            dnn = Dq

        w = 1.0 / (dnn + 1e-8) ** p
        w = w / w.sum(axis=1, keepdims=True)                 # (b, k)
        # 加权平均坐标
        Y[sl] = (w[..., None] * Y_fit[nn]).sum(axis=1).astype(np.float32)

    return Y
# ========== 主流程：按 subtask 学一个平面，再分模型面板 ==========
all_panels: Dict[Tuple[str,str], pd.DataFrame] = {}

for subtask, index_csv in INDEX_FILES.items():
    print(f"\n=== Subtask: {subtask} ===")
    idx = load_index(index_csv)

    rows = []
    for _, row in idx.iterrows():
        csv_path = str(row["csv_file_path"]).strip()
        df = read_csv_maybe(csv_path)
        if df is None or df.empty: 
            continue

        # 找列
        gen_col  = find_col(df, GEN_COL_CANDS, required=True)
        fit_col  = find_col(df, FITN_COL_CANDS, required=True)
        tour_col = find_col(df, TOUR_COL_CANDS, required=True)
        type_col = find_col(df, TYPE_COL_CANDS, required=False)
        model_col= find_col(df, MODEL_COL_CANDS, required=False)

        # 过滤 child
        if CHILD_ONLY and type_col and type_col in df.columns:
            df = df[df[type_col].astype(str).str.lower().eq("child")].copy()
            if df.empty: 
                continue

        # 拿模型名
        if model_col and model_col in df.columns and df[model_col].notna().any():
            model_name = str(df[model_col].dropna().iloc[0])
        else:
            model_name = None
            for c in MODEL_COL_CANDS:
                if c in row and isinstance(row[c], str) and row[c].strip():
                    model_name = row[c]; break
            if not model_name:
                model_name = Path(csv_path).stem

        # 取必要列
        sub = pd.DataFrame({
            "generation": pd.to_numeric(df[gen_col], errors="coerce"),
            "fitness":   pd.to_numeric(df[fit_col], errors="coerce"),
            "tour_raw":  df[tour_col],
            "model":     model_name,
        })
        if type_col and type_col in df.columns:
            sub["type"] = df[type_col].astype(str)
        else:
            sub["type"] = "unknown"
        sub = sub.dropna(subset=["generation","fitness","tour_raw"])

        # 解析 tour
        sub["tour"] = sub["tour_raw"].apply(coerce_tour)
        rows.append(sub)

    if not rows:
        print(f"[warn] no rows collected for {subtask}")
        continue

    big = pd.concat(rows, ignore_index=True)
    big = big.reset_index(drop=True)
    print(f"[info] collected rows: {len(big)} | models: {big['model'].nunique()}")

    # ---- 学任务级平面 ----
    # 抽样学 MDS（防爆），其余用 OOS 插值
    if len(big) <= MDS_MAX_POINTS:
        fit_idx = np.arange(len(big))
    else:
        fit_idx = stratified_sample_idx(big, per_bucket=PER_BUCKET, total_cap=MDS_MAX_POINTS)
    fit_mask = np.zeros(len(big), dtype=bool); fit_mask[fit_idx] = True

    tours = big["tour"].tolist()
    D_fit, E_fit, n_edges = pairwise_edge_distance([tours[i] for i in fit_idx])

    print(f"[mds] fitting on {len(fit_idx)} points (of {len(big)}) | edges per tour={n_edges}")
    mds = MDS(**MDS_KW)
    Y_fit = mds.fit_transform(D_fit)  # (m,2)

    # OOS：Shepard 加权插值（在“边差度量空间”做 KNN）
    Y = np.zeros((len(big), 2), dtype=np.float32)
    Y[fit_idx] = Y_fit

    if len(fit_idx) < len(big):
        # 预备：为加速，存 E_fit list
        not_idx = np.where(~fit_mask)[0]
        # 预先把所有 E（避免重复解析）
        E_all = [tour_edges_undirected(t) for t in tours]
        # 对每个查询，算到 Fit 的距离，再取 KNN，做权重平均
        for q in not_idx:
            di = distance_one_to_many_edges(E_all[q], E_fit, n_edges)
            if OOS_K < len(di):
                nn = np.argpartition(di, OOS_K)[:OOS_K]
            else:
                nn = np.arange(len(di))
            dnn = di[nn]
            w = 1.0 / (dnn + 1e-8)**OOS_P
            w = w / w.sum()
            Y[q] = (Y_fit[nn] * w[:,None]).sum(axis=0)

    # 写回
    big["mds_x"] = Y[:,0]; big["mds_y"] = Y[:,1]
    big["fitness_normed"] = robust_minmax(big["fitness"].to_numpy(), q=(1,99))

    # ---- 分模型建面板缓存 ----
    for model, dfm in big.groupby("model"):
        key = (subtask, str(model))
        all_panels.setdefault(key, pd.DataFrame())
        all_panels[key] = pd.concat([all_panels[key], dfm[["generation","fitness_normed","mds_x","mds_y","model"]]], ignore_index=True)

# ========== 画图：所有 (subtask, model) ==========
keys = sorted(all_panels.keys(), key=lambda x: (x[0], x[1].lower()))
if not keys:
    print("[warn] nothing to plot.")
else:
    n = len(keys)
    ncols = min(5, n); nrows = math.ceil(n/ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.8*nrows), squeeze=False)
    fig.suptitle("TSP — Task-level MDS (distance: 1 - |E∩|/n; color=generation; size=fitness)", fontsize=16, y=0.995)

    # 颜色映射（全局 generation）
    g_all = pd.concat([all_panels[k]["generation"] for k in keys], ignore_index=True)
    cmap = plt.get_cmap("viridis")
    gnorm = Normalize(vmin=g_all.min(), vmax=g_all.max())

    for i, key in enumerate(keys):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        dfp = all_panels[key].copy().sort_values("generation")

        cols  = cmap(gnorm(dfp["generation"].to_numpy(float)))
        sizes = 10 + 80 * dfp["fitness_normed"].to_numpy(float)

        # 底层密度（可选）
        # ax.hexbin(dfp["mds_x"], dfp["mds_y"], gridsize=40, mincnt=3, alpha=0.22)

        ax.scatter(dfp["mds_x"], dfp["mds_y"], s=sizes, c=cols,
                   edgecolor="white", linewidth=0.35, alpha=0.92, zorder=2)
        subtask, model = key
        ax.set_title(f"{subtask} | {model}", fontsize=10, pad=4)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

    # 关空轴
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r][c].set_axis_off()

    # Generation colorbar
    cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
    cb  = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gnorm); cb.set_label("Generation", rotation=90)

    # Size legend
    legend_sizes = [0.2, 0.5, 0.8]
    handles = [plt.scatter([], [], s=10+80*s, edgecolor="gray", facecolors="none", linewidth=0.8) for s in legend_sizes]
    axes[0][0].legend(handles, [f"fitness≈{s:.1f}" for s in legend_sizes],
                      scatterpoints=1, frameon=True, fontsize=8, loc="upper left")

    fig.subplots_adjust(left=0.06, right=0.90, top=0.93, bottom=0.06, wspace=0.12, hspace=0.12)
    plt.show()

In [ ]:
# ===== MDS 2D projection for BOTH TASKS, then per-(task, model) scatter =====
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict

# ======= 配置：指向两个任务的 parquet 文件 =======
TASK_PARQUETS: Dict[str, Path] = {
    "Summarization": Path("/Users/lievretre/Desktop/mds_pack/promptopt_sum_cos_coords.parquet"),
    "Simplification": Path("/Users/lievretre/Desktop/mds_pack/promptopt_sim_cos_coords.parquet"),
}

# ======= 第一步：加载、合并数据，并添加 'subtask' 列 =======
# all_dfs = []
sum_df = []
sim_df = []
# for subtask, in_path in TASK_PARQUETS.items():
#     if not in_path.exists():
#         print(f"[Warn] Missing parquet file, skipping: {in_path}")
#         continue
#     try:
#         df = pd.read_parquet(in_path)
#         df["subtask"] = subtask  # <-- 关键：添加任务标识
#         all_dfs.append(df)
#     except Exception as e:
#         print(f"[Error] Failed to read {in_path}: {e}")

# if not all_dfs:
#     raise ValueError("No data loaded. Please check TASK_PARQUETS paths.")


# all_df = pd.concat(all_dfs, ignore_index=True)
sim_df = pd.read_parquet(Path("/Users/lievretre/Desktop/mds_pack/promptopt_sim_cos_coords.parquet"))
sim_df["subtask"] = "Simplication"

sum_df = pd.read_parquet(Path("/Users/lievretre/Desktop/mds_pack/promptopt_sum_cos_coords.parquet"))
sum_df["subtask"] = "Summarization"


# ======= 第二步：在 *全部* 数据上进行统一归一化 =======
# generation: 使用全局 min/max
def fitness_norm(all_df):
    gen_norm = mpl.colors.Normalize(vmin=all_df["generation"].min(), vmax=all_df["generation"].max())
    
    # fitness: 使用全局鲁棒百分位
    def robust_minmax_reverse(x, q=(0, 100)):
        x_np = np.asanyarray(x)
        lo, hi = np.nanpercentile(x_np, q)
        den = (hi - lo) if hi > lo else 1.0
        return np.clip((hi - x_np) / den, 0, 1)
    
    all_df["fitness_normed"] = robust_minmax_reverse(all_df["fitness"])

    return all_df

sim_df = fitness_norm(sim_df)
sum_df = fitness_norm(sum_df)

all_df = pd.concat([sim_df,sum_df],ignore_index =True)


# ======= 第三步：设置面板 - 每个 (subtask, model) 一个子图 =======
# 1. 找出所有 (subtask, model) 组合键
keys = sorted(all_df.groupby(["subtask", "model"]).groups.keys())
n = len(keys)
if n == 0:
    raise ValueError("No (subtask, model) groups found in the data.")

# 2. 计算网格
ncols = min(5, n)
nrows = int(math.ceil(n / ncols))

# 3. 样式设置
mpl.rcParams.update(mpl.rcParamsDefault)
plt.style.use("default")
mpl.rcParams["figure.facecolor"] = "white"
mpl.rcParams["axes.facecolor"]   = "white"

fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.8 * nrows), squeeze=False)
cmap = plt.get_cmap("viridis")

# ======= 第四步：循环绘制所有子图 =======
for i, key in enumerate(keys):
    subtask, model_name = key  # <-- key 是 (subtask, model_name)
    r, c = divmod(i, ncols)
    ax = axes[r][c]

    # 筛选出当前子图的数据
    sub = all_df[(all_df["subtask"] == subtask) & (all_df["model"] == model_name)]
    if sub.empty:
        ax.set_title(f"{subtask} | {model_name}\n(No data)", fontsize=9, pad=4)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        continue

    # 颜色和尺寸（使用已归一化的列）
    cols = cmap(gen_norm(sub["generation"].to_numpy(float)))
    sizes = 10 + 80 * sub["fitness_normed"].to_numpy(float)

    # 背景密度（可选）
    try:
        # 使用 mds 坐标
        ax.hexbin(sub["mds_x"], sub["mds_y"], gridsize=35, cmap='Greys', mincnt=3, alpha=0.15, zorder=1)
    except Exception:
        print(f"hexbin failed for model {key}") # 可选：取消注释以进行调试
        pass

    # 散点图
    ax.scatter(sub["mds_x"], sub["mds_y"], s=sizes, c=cols,
               edgecolor="white", linewidth=0.35, alpha=0.92, zorder=2)

    # --- 关键：修改标题格式 ---
    title_model = model_name.replace("_np2_nc5_simple", "").replace(":free", "").replace("-instruct", "")
    # 找到最后一个 "_" 的位置
    last_underscore_index = title_model.rfind("_")
    
    # 从最后一个 "_" 开始截取模型名称
    
    title_model = title_model[last_underscore_index + 1:]

    ax.set_title(f"{subtask} | {title_model}", fontsize=10, pad=4)
    
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

# 关掉多余的空轴
for j in range(n, nrows * ncols):
    r, c = divmod(j, ncols)
    axes[r][c].set_axis_off()

# ======= 第五步：全局标题、颜色条和图例 =======
fig.suptitle("Prompt Optimization (Simplification / Summarization) — Novelty at the Semantic Level", fontsize=15, y=0.995)

# 颜色条 (Generation)
cax = fig.add_axes([0.92, 0.20, 0.015, 0.60])
cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=gen_norm)
cb.set_label("Generation", rotation=90)

# # 尺寸图例 (Fitness)
# legend_sizes = [0.2, 0.5, 0.8]
# handles = [plt.scatter([], [], s=10 + 80 * s, edgecolor="gray", facecolors="none", linewidth=0.8)
#            for s in legend_sizes]
# # 将图例放在第一个子图的左上角
# axes[0][0].legend(handles, [f"fitness≈{s:.1f}" for s in legend_sizes],
#                   scatterpoints=1, frameon=True, fontsize=8, loc="upper left",
#                   bbox_to_anchor=(0.01, 0.99), borderaxespad=0.)
# 尺寸图例 (Fitness)
legend_sizes = [0.2, 0.5, 0.8]
handles = [plt.scatter([], [], s=10 + 80 * s, edgecolor="gray", facecolors="none", linewidth=0.8)
           for s in legend_sizes]
labels = [f"fitness≈{s:.1f}" for s in legend_sizes] # 先定义 labels

# --- ✅ 修改：将图例放在整个大图的右上角区域 ---
# (我们将其放置在 Colorbar 的正上方)
fig.legend(handles, labels,
           scatterpoints=1, 
           frameon=True, 
           fontsize=10, 
           loc="upper left",  # 将图例的“左上角”...
           bbox_to_anchor=(0.90, 0.93), # ...锚定到 (x=92%, y=90%) 的位置
           # title="Fitness", # 可以给图例加个标题
          )

fig.subplots_adjust(left=0.06, right=0.90, top=0.94, bottom=0.06, wspace=0.12, hspace=0.20) # 增加了 hspace
plt.savefig("/Users/lievretre/Desktop/evo_eval_plots_10_28/novelty_trajectory_plots/novelty_mds_promptopt.png",dpi=300)
plt.show()